In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Natural Language Processing with Disaster Tweets

![](https://www.kaggle.com/competitions/17777/images/header)

* Twitter has become an important communication channel in times of emergency.The ubiquitousness of smartphones enables people to announce an emergency they’re observing in real-time. Because of this, more agencies are interested in programatically monitoring Twitter (i.e. disaster relief organizations and news agencies).
* In this project tweets will be analyzed to predict which Tweets are about real disasters and which one’s aren’t.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
train=pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test=pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')

In [ ]:
train.head()

In [ ]:
train.location.unique()

In [ ]:
train.shape,test.shape

In [ ]:
test.head()

In [ ]:
import matplotlib.pyplot as plt

value_counts = train['target'].value_counts()
plt.figure(figsize=(8, 6))
value_counts.plot(kind='bar', color=['blue', 'green']) 
plt.title('Value Counts of Target Column')
plt.xlabel('Target')
plt.ylabel('Count')
plt.xticks(rotation=0)  
plt.show()



In [ ]:
train=train[["text","target"]] 

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
df = pd.concat([train, test], ignore_index=True)

In [ ]:
df.head()

In [ ]:
df=df[["text","target"]] 

In [ ]:
df.isnull().sum()

In [ ]:
pip install langdetect

In [ ]:
from langdetect import detect
from collections import Counter
import pandas as pd

# Dil tespit fonksiyonu (hataları yakalar)
def detect_language_safe(text):
    try:
        return detect(text)
    except:
        return "error"

# Dil sütununu oluştur
test['lang'] = test['text'].astype(str).apply(detect_language_safe)

# Dilleri say
language_counts = test['lang'].value_counts()

# Sonuçları yazdır
print(language_counts)


In [ ]:
pip install googletrans==4.0.0-rc1


In [ ]:
from langdetect import detect
from googletrans import Translator
import pandas as pd

translator = Translator()

def translate_to_english(text):
    try:
        lang = detect(text)
        if lang != 'en':
            translated = translator.translate(text, src=lang, dest='en')
            return translated.text
        else:
            return text
    except Exception as e:
        print(f"Error for text: {text} -> {e}")
        return text



df['text'] = df['text'].apply(translate_to_english)

#print(df)


In [ ]:
df['text']=df['text'].str.lower() 

In [ ]:
df['text']=df['text'].str.replace('[^\w\s]','')
df['text']=df['text'].str.replace('\n','')
df['text']=df['text'].str.replace('\d+','')
df['text']=df['text'].str.replace('\r',' ')

In [ ]:
import re
import string
import nltk

def tokenization(text):
    text = re.split('\W+', text)
    return text

df['tokenized'] = df['text'].apply(lambda x: tokenization(x.lower()))
df.head()

In [ ]:
import nltk
nltk.download('stopwords')
stopword = nltk.corpus.stopwords.words('english')

In [ ]:
def remove_stopwords(text):
    text = [word for word in text if word not in stopword]
    return text
    
df['nonstop'] = df['tokenized'].apply(lambda x: remove_stopwords(x))
df.head(10)

In [ ]:
ps = nltk.PorterStemmer()

def stemming(text):
    text = [ps.stem(word) for word in text]
    return text

df['stemmed'] = df['nonstop'].apply(lambda x: stemming(x))
df.head()

In [ ]:
def clean_text(text):
    text_lc = "".join([word.lower() for word in text if word not in string.punctuation]) # remove puntuation
    text_rc = re.sub('[0-9]+', '', text_lc)
    tokens = re.split('\W+', text_rc)    # tokenization
    text = [ps.stem(word) for word in tokens if word not in stopword]  # remove stopwords and stemming
    return text

In [ ]:
df['stemmedtext'] = df['stemmed'].apply(lambda x: ' '.join(x))

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
from nltk.stem import PorterStemmer
pr=PorterStemmer()

In [ ]:
def lemmafn(text):
    words=TextBlob(text).words
    return[pr.stem(word) for word in words]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vect=CountVectorizer(stop_words='english',ngram_range=(1,2),max_features=10000,analyzer=lemmafn)

In [ ]:
vect

In [ ]:
train.sample(10)

# Model

In [ ]:
train = df.iloc[:7613]
test = df.iloc[7613:]

x_train=train.stemmedtext
y_train=train['target']

x_test=test.stemmedtext
y_test=test['target']

from textblob import TextBlob
x_train=vect.fit_transform(x_train)

In [ ]:
def fnc_classification_all_model(x,y):
    from sklearn.naive_bayes import GaussianNB
    from sklearn.naive_bayes import BernoulliNB
    from sklearn.svm import SVC
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import GradientBoostingClassifier
    from xgboost import XGBClassifier
    from sklearn.model_selection import train_test_split
    
    from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
    from sklearn.metrics import confusion_matrix,classification_report
              
    g=GaussianNB()
    b=BernoulliNB()
    D=DecisionTreeClassifier()
    R=RandomForestClassifier()
    Log=LogisticRegression()
    XGB=XGBClassifier()
    G=GradientBoostingClassifier()
      
    x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42)
    
    
    algos=[b,D,R,Log,XGB,G]
    algo_names=['BernoulliNB','DecisionTreeClassifier','RandomForestClassifier','LogisticRegression','XGBClassifier','GradientBoostingClassifier']
    
    accuracy_scored=[]
    precision_scored=[]
    recall_scored=[]
    f1_scored=[]
       
    
    for item in algos:
        print(item)

        predict=item.fit(x_train,y_train).predict(x_test)
        
        
        accuracy_scored.append(accuracy_score(y_test,predict))
        precision_scored.append(precision_score(y_test,predict,average='macro'))
        recall_scored.append(recall_score(y_test,predict,average='macro'))
        f1_scored.append(f1_score(y_test,predict,average='macro'))

    result=pd.DataFrame(columns=['accuracy_score','f1_score','recall_score','precision_score'],index=algo_names)
    result['accuracy_score']=accuracy_scored
    result['f1_score']=f1_scored
    result['recall_score']=recall_scored
    result['precision_score']=precision_scored
    
    return result.sort_values('accuracy_score',ascending=False)

In [ ]:
fnc_classification_all_model(x_train,y_train)

In [ ]:
from sklearn.naive_bayes import BernoulliNB
b=BernoulliNB()

In [ ]:
b.fit(x_train,y_train)

In [ ]:
x_test=vect.fit_transform(x_test) #x_testi de vektörize ettik
pred=b.predict(x_test)

In [ ]:
pred

In [ ]:
submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
submission[:5]

In [ ]:
submission['target'] = pred.astype(int) # arrayi int veri tipine çevirip target sütununa atadık

In [ ]:
submission.head()

In [ ]:
submission.to_csv("submissionDisaster.csv", index=False)

* In this project tweets are analyzed to predict which Tweets are about real disasters and which one’s aren’t. 
* Bernoulli NB algorithm is chosen according to training performance. 